In [1]:
import joblib

objetos_modelagem = joblib.load(
    "../data/objetos_modelagem.pkl"
)

print(objetos_modelagem.keys())

dict_keys(['X_train', 'X_test', 'y_train', 'y_test', 'preprocessor', 'features_selecionadas'])


In [2]:
import sys
from pathlib import Path

# Adiciona a raiz do projeto ao PATH, caso necessário
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

from src.modeling.modeling import (
    treinar_decision_tree,
    treinar_random_forest,
    treinar_decision_tree_randomized,
    treinar_random_forest_randomized,
    treinar_xgboost_randomized,
    avaliar_modelo,
    comparar_modelos
)

In [3]:
X_train = objetos_modelagem["X_train"]
X_test = objetos_modelagem["X_test"]

y_train = objetos_modelagem["y_train"]
y_test = objetos_modelagem["y_test"]

preprocessor = objetos_modelagem["preprocessor"]

features_selecionadas = objetos_modelagem["features_selecionadas"]

In [4]:
# Treinar Decision Tree (já roda a validação cruzada internamente)
modelo_dt, grid_dt = treinar_decision_tree(
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)


===== VALIDAÇÃO CRUZADA - Decision Tree (parâmetros padrão) =====
Scores por fold (f1): [0.6242 0.627  0.6261 0.6247 0.6247]
Média:         0.6253
Desvio padrão: 0.0010
Fitting 5 folds for each of 120 candidates, totalling 600 fits

===== DECISION TREE (após GridSearch) =====
Melhores parâmetros:
{'criterion': 'entropy', 'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 2}

Melhor score (média da validação cruzada):
0.6605077406804771


In [5]:
# Treinar Random Forest
modelo_rf, grid_rf = treinar_random_forest_randomized(
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)


===== VALIDAÇÃO CRUZADA - Random Forest (parâmetros padrão) =====
Scores por fold (f1): [0.6251 0.626  0.626  0.6257 0.6239]
Média:         0.6253
Desvio padrão: 0.0008
Fitting 3 folds for each of 8 candidates, totalling 24 fits

===== RANDOM FOREST (após RandomizedSearch) =====
Melhores parâmetros:
{'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 5, 'max_depth': 15}

Melhor score (média da validação cruzada):
0.6290046001238644


In [6]:
# Treinar XGBoost
modelo_xg, grid_xg = treinar_xgboost_randomized(
    X_train,
    y_train,
    cv=5,
    scoring="f1"
)


===== VALIDAÇÃO CRUZADA - XGBoost (parâmetros padrão) =====
Scores por fold (f1): [0.625  0.6283 0.6284 0.6288 0.6295]
Média:         0.6280
Desvio padrão: 0.0016
Fitting 5 folds for each of 15 candidates, totalling 75 fits

===== XGBOOST (após RandomizedSearch) =====
Melhores parâmetros:
{'subsample': 0.7, 'reg_lambda': 1.5, 'reg_alpha': 0.1, 'n_estimators': 300, 'min_child_weight': 5, 'max_depth': 7, 'learning_rate': 0.05, 'gamma': 0.3, 'colsample_bytree': 0.8}

Melhor score (média da validação cruzada):
0.6277224407744452


In [7]:
# ============================================================
# AVALIAÇÃO DOS MODELOS
# ============================================================


# --------------------------------------------------------
# Decision Tree
# --------------------------------------------------------
resultado_dt_treino = avaliar_modelo(
    modelo_dt,
    X_train,
    y_train,
    nome_modelo="Decision Tree - Treino"
)

resultado_dt_teste = avaliar_modelo(
    modelo_dt,
    X_test,
    y_test,
    nome_modelo="Decision Tree - Teste"
)

# --------------------------------------------------------
# Random Forest
# --------------------------------------------------------
resultado_rf_treino = avaliar_modelo(
    modelo_rf,
    X_train,
    y_train,
    nome_modelo="Random Forest - Treino"
)

resultado_rf_teste = avaliar_modelo(
    modelo_rf,
    X_test,
    y_test,
    nome_modelo="Random Forest - Teste"
)


# --------------------------------------------------------
# XGBoost
# --------------------------------------------------------
resultado_xg_treino = avaliar_modelo(
    modelo_xg,
    X_train,
    y_train,
    nome_modelo="XGBoost - Treino"
)

resultado_xg_teste = avaliar_modelo(
    modelo_xg,
    X_test,
    y_test,
    nome_modelo="XGBoost - Teste"
)

# --------------------------------------------------------
# Comparação consolidada
# --------------------------------------------------------
comparacao_final = comparar_modelos([
    resultado_dt_treino,
    resultado_dt_teste,
    resultado_rf_treino,
    resultado_rf_teste,
    resultado_xg_treino,
    resultado_xg_teste
])

comparacao_final


===== Decision Tree - Treino =====
Accuracy:  0.6048
Precision: 0.6037
Recall:    0.7292
F1-Score:  0.6605
ROC-AUC:   0.6375

Matriz de confusão:
[[312937 358383]
 [202731 545857]]

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.47      0.53    671320
           1       0.60      0.73      0.66    748588

    accuracy                           0.60   1419908
   macro avg       0.61      0.60      0.59   1419908
weighted avg       0.61      0.60      0.60   1419908


===== Decision Tree - Teste =====
Accuracy:  0.6043
Precision: 0.6034
Recall:    0.7277
F1-Score:  0.6598
ROC-AUC:   0.6365

Matriz de confusão:
[[ 78325  89505]
 [ 50958 136190]]

Classification Report:
              precision    recall  f1-score   support

           0       0.61      0.47      0.53    167830
           1       0.60      0.73      0.66    187148

    accuracy                           0.60    354978
   macro avg       0.60      0.60      0.59 

,modelo,accuracy,precision,recall,f1,roc_auc
0,Decision Tree - Treino,0.604824,0.603664,0.729182,0.660513,0.637538
1,Decision Tree - Teste,0.604305,0.603425,0.727713,0.659767,0.636462
2,Random Forest - Treino,0.625578,0.656388,0.608177,0.631363,0.678243
4,XGBoost - Treino,0.623747,0.654982,0.605044,0.629023,0.675256
3,Random Forest - Teste,0.621901,0.652949,0.603709,0.627364,0.672655
5,XGBoost - Teste,0.620393,0.651718,0.601321,0.625506,0.670769


In [8]:
# ============================================================
# SALVAR MODELOS TREINADOS
# ============================================================

joblib.dump(modelo_dt, "../data/modelo_decision_tree.pkl")
joblib.dump(modelo_rf, "../data/modelo_random_forest.pkl")
joblib.dump(modelo_xg, "../data/modelo_xgboost.pkl")

print("Modelos salvos com sucesso em ../data/")

Modelos salvos com sucesso em ../data/
